In [1]:
import sys
import os
from pathlib import Path
project_dir = Path(os.path.abspath('')).parent
sys.path.insert(0, project_dir.as_posix())

from tqdm import trange, tqdm
import numpy as np
import torch
from matplotlib import pyplot as plt

%load_ext autoreload
%autoreload 2
import train_dpr

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
%matplotlib inline

In [2]:
args = train_dpr.TrainArgs(
    save_weight="dpr_bert7",
    epochs=1,
    batch_size=64,
    accumulation_steps=1,
    max_query_len=64,
    max_passage_len=512,
    learning_rate=4e-6,
    use_amp=True,
    from_weight="none",
    train_bert=True,
    use_wandb=True,
    wandb_entity="ztzhu11",
    dtype="float16",
)
model, tokenizer, train_ds = train_dpr.train(args, early_return=1)


所加载Model可训练参数：11.171 百万


In [3]:
with torch.inference_mode():
    queries = train_ds.dataset["query"][:64]
    answers = train_ds.dataset["answer"][:64]
    query_inputs = tokenizer(queries, max_length=args.max_query_len, truncation=True, return_tensors="pt", padding_side="right", padding="max_length", return_token_type_ids=False).to(device)
    answer_inputs = tokenizer(answers, max_length=args.max_query_len, truncation=True, return_tensors="pt", padding_side="right", padding="max_length", return_token_type_ids=False).to(device)
    loss, similarity = model(query_inputs, answer_inputs)
    print(loss)
    print(train_dpr.calc_relative_advantage(similarity).mean())
    print(similarity)
    print(train_dpr.calc_relative_advantage(similarity))

tensor(3.9810, device='cuda:0')
tensor(-0.0399, device='cuda:0')
tensor([[0.5616, 0.4069, 0.5041,  ..., 0.3412, 0.3415, 0.4694],
        [0.4569, 0.5654, 0.5289,  ..., 0.4543, 0.3068, 0.5023],
        [0.3234, 0.3288, 0.7045,  ..., 0.2108, 0.4865, 0.4609],
        ...,
        [0.4194, 0.5371, 0.4367,  ..., 0.5323, 0.1750, 0.3880],
        [0.4064, 0.3082, 0.6446,  ..., 0.2246, 0.7066, 0.4759],
        [0.5148, 0.4366, 0.4510,  ..., 0.3134, 0.3202, 0.6793]],
       device='cuda:0')
tensor([-0.1117, -0.2014,  0.0729, -0.1060, -0.3088, -0.1072, -0.1410,  0.1577,
        -0.4558, -0.0088,  0.0736,  0.0236,  0.0000, -0.2828,  0.0272,  0.1957,
         0.0321,  0.0000, -0.3294,  0.1242, -0.2062, -0.3171, -0.1782,  0.0000,
         0.1139,  0.0000, -0.4104, -0.2726,  0.2161,  0.1354, -0.1141,  0.0909,
         0.0559, -0.0415,  0.1599, -0.1028,  0.1076,  0.1798,  0.0304,  0.0000,
         0.0672, -0.0419,  0.1174,  0.0401, -0.1779, -0.0163, -0.2877,  0.0399,
        -0.0854, -0.3394,  0.1912

In [4]:
train_dpr.train(args, model, tokenizer, train_ds)

wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: ztzhu1 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


 16%|█▌        | 100/627 [00:38<03:27,  2.54it/s, [1/1]loss=3.7005,adv=0.3040,grad_norm=1.2010]

Epoch:[1/1,100/627] loss:3.7005


 32%|███▏      | 200/627 [01:20<03:09,  2.25it/s, [1/1]loss=3.6782,adv=0.3245,grad_norm=1.1391]

Epoch:[1/1,200/627] loss:3.6782


...

 96%|█████████▌| 600/627 [03:28<00:07,  3.72it/s, [1/1]loss=3.6098,adv=0.3523,grad_norm=1.1468]

Epoch:[1/1,600/627] loss:3.6098


 96%|█████████▋| 604/627 [03:30<00:08,  2.86it/s, [1/1]loss=3.6153,adv=0.2952,grad_norm=1.1826]Traceback (most recent call last):
  File "/workspace/minimind/trainer_ztzhu/train_dpr.py", line 691, in train
    spend_time = train_epoch(
                 ^^^^^^^^^^^^
  File "/workspace/minimind/trainer_ztzhu/train_dpr.py", line 472, in train_epoch
    torch.cuda.empty_cache()
  File "/root/.pyenv/versions/3.11.1/lib/python3.11/site-packages/torch/cuda/memory.py", line 224, in empty_cache
    torch._C._cuda_emptyCache()
KeyboardInterrupt


train/adv,▁▂▁▂▃▃▄▃▅▄▅▅▅▆▇▇▆█▅▆▆▆▇▆█▇▆▇▆▆▆▆▇▇▇▇▇▇▅▆
train/epoch,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/grad_norm,█▅▅▅▅▃▄▃▂▂▂▂▂▂▂▃▂▁▁▂▂▂▂▅▂▁▂▁▂▂▂▂▂▁▁▁▂▂▁▂
train/loss,▇█▄▄▃▄▄▄▄▃▃▃▃▂▃▃▂▂▂▂▂▂▂▁▂▂▂▂▂▂▂▂▁▁▂▁▂▂▂▂
train/lr,██████▇▇▇▇▆▆▆▆▆▅▅▅▄▄▄▄▄▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁
train/time,▁▁▂▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇██████
train_step,▁▁▁▁▁▁▁▂▂▂▂▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇█████
train/adv,0.29517
train/epoch,1
train/grad_norm,1.18256
train/loss,3.61533


KeyboardInterrupt: 

In [6]:
with torch.inference_mode():
    queries = train_ds.dataset["query"][1000:1030]
    answers = train_ds.dataset["answer"][1000:1030]
    query_inputs = tokenizer(queries, max_length=args.max_query_len, truncation=True, return_tensors="pt", padding_side="right", padding="max_length", return_token_type_ids=False).to(device)
    answer_inputs = tokenizer(answers, max_length=args.max_query_len, truncation=True, return_tensors="pt", padding_side="right", padding="max_length", return_token_type_ids=False).to(device)
    loss, similarity = model(query_inputs, answer_inputs)
    print(loss)
    print(train_dpr.calc_relative_advantage(similarity).mean())
    print(similarity)
    print(train_dpr.calc_relative_advantage(similarity))

tensor(2.8905, device='cuda:0')
tensor(0.5257, device='cuda:0')
tensor([[ 5.3881e-01, -1.2595e-01,  4.0893e-01, -1.8024e-01,  1.8517e-01,
         -2.3428e-02,  4.9280e-01,  2.3676e-01, -2.0166e-01,  1.5778e-01,
          5.3208e-01, -3.1813e-02, -5.4580e-02,  1.0599e-01,  8.0236e-02,
         -4.8122e-02, -1.7655e-01,  4.6926e-01,  2.5154e-01, -1.8762e-02,
         -1.1359e-01, -7.8144e-02, -2.2065e-01,  3.6821e-01,  1.4812e-01,
         -6.2643e-02, -1.7130e-01, -1.9588e-01,  1.7515e-01,  2.2665e-02],
        [-1.1378e-01,  5.7797e-01,  1.6621e-01,  1.5080e-01, -1.5776e-02,
          1.5961e-01,  1.1736e-01,  1.5613e-01, -1.5526e-01,  3.7879e-02,
          1.7958e-03,  3.0814e-01,  1.4356e-01,  2.8752e-03, -6.3452e-02,
          7.4637e-03,  1.9540e-01,  1.8858e-01,  1.5018e-01, -1.1376e-01,
          9.7223e-02,  1.3839e-01,  2.3447e-01,  2.0381e-01,  1.3111e-01,
          1.5008e-01, -7.6586e-02, -5.7355e-02,  6.9425e-02,  2.5086e-01],
        [ 5.1431e-01, -2.2104e-01,  5.1430e-01

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

In [ ]:
with torch.inference_mode():
    queries = train_ds.dataset["query"][:64]
    answers = train_ds.dataset["answer"][:64]
    q_reps = model.encode(queries)
    a_reps = model.encode(answers)
    similarity = model.similarity(q_reps, a_reps)
    print(train_dpr.calc_relative_advantage(similarity).mean())
    print(similarity)
    print(train_dpr.calc_relative_advantage(similarity))

tensor(0.8158)
tensor([[ 0.7186,  0.1013, -0.0791,  ...,  0.1400,  0.0855,  0.0940],
        [ 0.2992,  0.6470, -0.0567,  ...,  0.3474, -0.0198,  0.2313],
        [-0.0365, -0.0444,  0.5183,  ..., -0.0338,  0.3380, -0.0294],
        ...,
        [ 0.2689,  0.2214, -0.1140,  ...,  0.6362, -0.0036,  0.0679],
        [ 0.1681, -0.0308,  0.1020,  ..., -0.0807,  0.5542, -0.0236],
        [ 0.1117,  0.0870, -0.0921,  ...,  0.1372,  0.0211,  0.7237]])
tensor([ 5.5203e-01,  7.8414e-01,  1.6541e-01,  5.6067e-01,  5.6038e-01,
         4.7162e-01,  5.4036e-01,  7.0732e-01,  9.6240e-01,  3.0614e-01,
         2.4259e-01,  6.2544e-01,  2.2311e-01,  1.4136e-01,  7.2277e-01,
         4.7744e-01,  1.3562e+00,  1.0552e+00,  4.5371e-01,  5.7140e-01,
         6.6375e-02,  0.0000e+00, -6.6656e-02,  1.3249e+00,  7.5039e-01,
         3.3052e-02,  6.2327e-01,  8.0180e-01,  6.9358e-01,  1.7268e+00,
         2.7855e-01,  2.1614e+00,  1.5603e+00,  1.4302e+00,  3.0825e-01,
         7.3917e-01,  2.3460e+00,  9.145